# Setup: turn a stock PyTorch image into the repo-intel GPU image

Run this **once**, in a Jupyter job on the SVKM cluster. At the end you save the
container as your own image and never run this again — later jobs start from the
saved image and go straight to work.

**Start the job like this** in Altair Access → Applications → Jupyter:

| Field | Value |
|---|---|
| Container Image | `pytorch_pbs:23.06-py3` |
| Number of Nodes | 1 |
| Number of Processors per Node | **8** (not 1) |
| Number of GPUs per Node | 1 |
| Amount of Memory (MB) | **32000** (not 10) |
| Queue | `workq` |

The two bolded fields default to values that will kill the job: 1 CPU and **10 MB**
of RAM. Set them before submitting.


## 1. What did we actually get?

Run this first. It answers the three things the manual does not: how much GPU we
were given, whether this node can reach the internet, and where to put files.


In [ ]:
import os, shutil, socket, subprocess, sys

def sh(cmd):
    try:
        return subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=60).stdout.strip()
    except Exception as e:
        return f'(failed: {e})'

print('=== GPU ===')
print(sh('nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader') or '(no nvidia-smi)')
mig = sh("nvidia-smi --query-gpu=mig.mode.current --format=csv,noheader")
print('MIG mode:', mig or '(unknown)')
print(sh('nvidia-smi -L'))

print('\n=== CPU / RAM / disk ===')
print('cores visible :', os.cpu_count())
mem = sh("grep MemTotal /proc/meminfo")
print('host memory   :', mem)
for p in ('/data', '/tmp', os.path.expanduser('~')):
    if os.path.isdir(p):
        t, u, f = shutil.disk_usage(p)
        print(f'{p:<8} free {f/2**30:6.1f} GB of {t/2**30:6.1f} GB')

print('\n=== network (decides whether we can pull models here) ===')
for host, port, label in [('pypi.org',443,'PyPI (pip)'),
                          ('github.com',443,'GitHub (git clone)'),
                          ('registry.ollama.ai',443,'Ollama model registry')]:
    try:
        socket.create_connection((host, port), timeout=6).close()
        print(f'  REACHABLE  {label}')
    except Exception as e:
        print(f'  BLOCKED    {label}  ({type(e).__name__})')


### How to read that

- **GPU** should show an H100 and, under MIG, a ~40 GB slice. 40 GB is plenty: a 7B
  4-bit model needs about 6 GB, so it fits entirely in VRAM with no CPU offload —
  which is where nearly all the speedup over the laptop comes from.
- **Ollama model registry REACHABLE** → you can pull models here (§4).
  **BLOCKED** → you must upload the weights instead; §4 explains that path.
- **`/data`** is your persistent storage and survives between jobs. `/tmp` usually
  does not.


## 2. Install Ollama

We keep Ollama rather than running the models through PyTorch directly. It serves
**4-bit quantized** weights, which is what every result so far was produced with.
Switching to fp16 would use the H100 better and produce *different text from the
same model* — a different experiment, under which none of the existing 157 summaries
stay comparable. See `docs/GPU_BATCH.md`.


In [ ]:
# The installer needs GitHub. If §1 showed GitHub BLOCKED, skip to the manual
# fallback in the next cell instead.
!curl -fsSL https://ollama.com/install.sh | sh
!ollama --version


In [ ]:
# FALLBACK, only if the installer could not reach GitHub.
# Download the tarball somewhere with internet, upload it via Altair's +Upload into
# /data/<you>/, then run this.
#
# !tar -C /usr -xzf /data/$USER/ollama-linux-amd64.tgz
# !ollama --version


## 3. Python dependencies and the project code

Upload the repository as a zip through Altair (**+Upload**) into `/data/<you>/`, or
clone it if §1 showed GitHub is reachable.


In [ ]:
import os, subprocess
USER = os.environ.get('USER', 'mpstme-admin')
DATA = f'/data/{USER}'
PROJ = f'{DATA}/repo-intel-platform'
print('project dir ->', PROJ)

if not os.path.isdir(PROJ):
    print('Not found. Either:')
    print(f'  git clone https://github.com/palsoniii/repo-intel-platform {PROJ}')
    print(f'  ...or upload a zip to {DATA} and: unzip repo-intel-platform.zip -d {DATA}')
else:
    print('found')


In [ ]:
# The image already has torch. Install the rest, and do NOT let pip replace torch
# with a different build.
!pip install --no-cache-dir -r {PROJ}/backend/requirements.txt
!python -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"


## 4. Stage the models

About 15 GB for all three. Put them on `/data` so they persist between jobs and are
**not** baked into the saved image — an image with weights inside would be enormous
and would need rebuilding every time a model changes.


In [ ]:
import os
os.environ['OLLAMA_MODELS'] = f'{DATA}/ollama'
os.makedirs(os.environ['OLLAMA_MODELS'], exist_ok=True)

# Start the server in the background; it must be running for pull/run to work.
!pkill -f 'ollama serve' 2>/dev/null; sleep 1
get_ipython().system_raw('OLLAMA_MODELS=' + os.environ['OLLAMA_MODELS'] + ' nohup ollama serve > /tmp/ollama.log 2>&1 &')
import time; time.sleep(5)
!ollama list || tail -20 /tmp/ollama.log


In [ ]:
# If the Ollama registry was BLOCKED in §1, skip this cell. Instead pull these on a
# machine with internet, then upload that whole directory to $DATA/ollama.
for m in ['qwen2.5-coder:7b', 'codellama:7b-instruct', 'gemma2:9b']:
    print(f'--- pulling {m} ---')
    !ollama pull {m}
!ollama list


## 5. Smoke test

One generation, end to end. If this works, the image is good.


In [ ]:
import time, os
os.environ['OLLAMA_HOST'] = 'http://127.0.0.1:11434'
os.environ['OLLAMA_NUM_CTX'] = '8192'   # must match the study, or nothing is comparable

t0 = time.time()
!ollama run qwen2.5-coder:7b "Reply with exactly: OK" --verbose
print(f'\nwall clock: {time.time()-t0:.1f}s')
print('\nGPU during/after the call — memory should be in use, not 0:')
!nvidia-smi --query-gpu=memory.used,utilization.gpu --format=csv,noheader


### The number that matters

On the RTX 3050 with CPU offload, a real summary averaged **88 seconds**. Fully
resident on a 40 GB slice, expect roughly **5–15× faster**. If your smoke test is
slow and `nvidia-smi` shows little memory in use, the model is running on CPU —
stop and fix that before running a battery, or you will burn hours for nothing.


## 6. Save the container

Back in **Altair Access**:

1. Note this job's ID (e.g. `pbs.1325.srv-svkmmastermum.x8z`)
2. **Custom Actions → Save Docker Container**
3. **Job ID**: this job · **New Image Name**: `repo-intel-gpu` · **Current Working Directory**: `/data`
4. **Run**

Future jobs set *Container Image* to `repo-intel-gpu` and skip this whole notebook.

**Do not put the models or the repo inside the image.** They live on `/data`, which
every job can see. Baking them in makes the image tens of GB and stale the moment
either changes.
